# Pt(100) + CO surface adsorption study
# 铂(100)表面一氧化碳吸附研究

**System / 体系:** CO on Pt(100)  
**Functional / 泛函:** PBE (fixed for the whole study)  
**Guide / 方法参考:** class notebook `Castep_adsorption_study.ipynb` (method only — do not paste)

This is the **working study notebook** (not the blank teaching scaffold).
Parts 0–2 include complete job/analysis code with documented starting choices.
Parts 3–6 implement the remaining study and all extensions.
After each convergence job, **update** the production settings in later cells if your plots demand it.

这是**可运行的研究笔记本**。收敛扫描跑完后，若曲线要求更严/更松，请改后面单元格里的生产参数。

**Before anything else / 首先：** fill `NAME` and `USERNAME` in the next cell.


In [ ]:
# Cell 1: identity + system (Pt + CO)
# 姓名/用户名必须填写后才能继续

NAME      = "____"   # <-- fill / 填写你的姓名
USERNAME  = "____"   # <-- fill / 填写集群用户名
METAL     = "Pt"
ADSORBATE = "CO"
# Experimental FCC Pt lattice constant (Angstrom). Cite source in Part 1.3 notes.
# 实验晶格常数初值；在 1.3 文字说明里写出来源（如 CRC / 教材）。
A0_GUESS  = 3.924

assert NAME != "____" and USERNAME != "____", "Fill NAME and USERNAME first"
assert METAL == "Pt" and ADSORBATE == "CO"

with open("system.txt", "w") as f:
    f.write(f"{METAL} {ADSORBATE} {A0_GUESS}\n")
print(f"OK: {NAME} ({USERNAME}) — {ADSORBATE} on {METAL}(100); system.txt written.")


## Part 0: Setup / 准备工作

Run from your **home directory** study folder (not `classfiles`). One directory per study.


In [ ]:
# Environment checks
import os, pathlib
here = pathlib.Path.cwd()
assert "classfiles" not in str(here), "Copy this notebook out of classfiles first"
print(f"OK: working in {here}")

kw = pathlib.Path.home() / ".config" / "ase" / "castep_keywords.json"
print(f"OK: keyword cache present: {kw.exists()}")

import ase
print(f"OK: ase {ase.__version__}")


## Part 1: Bulk Pt / 体相铂

### 1.1 Cutoff convergence / 截断能收敛

Starting sweep: 300–700 eV in steps of 50 eV (refine if the curve has not settled).


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J pt_cutoff
#!/usr/bin/python3
# Job 1.1: Pt bulk cutoff sweep
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import bulk
from ase.calculators.castep import Castep

metal, adsorbate, a0 = open('system.txt').read().split()
a0 = float(a0)

cutoffs = list(range(300, 701, 50))

atoms = bulk(metal, 'fcc', a=a0, cubic=True)

with open('results_cutoff.txt', 'w') as f:
    for c in cutoffs:
        calc = Castep(directory=f'bulk_cut_{c}', label=f'bulk_cut_{c}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = c
        calc.param.task = 'SinglePoint'
        calc.cell.kpoint_mp_grid = '6 6 6'
        atoms.calc = calc
        e = atoms.get_potential_energy()
        n = len(atoms)
        f.write(f"{c} {e/n}\n")
        f.flush()
        print(f"cutoff {c} eV -> {e/n:.6f} eV/atom", flush=True)
print("done")


In [ ]:
# Analyse cutoff sweep — choose production cutoff from the plot
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('results_cutoff.txt')
cut, e = data[:, 0], data[:, 1]
de_meV = np.diff(e) * 1000

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(cut, e, 'o-')
ax1.set_xlabel('cutoff (eV)')
ax1.set_ylabel('energy (eV/atom)')
ax2.semilogy(cut[1:], np.abs(de_meV), 'o-')
ax2.axhline(5, ls='--', c='r', label='5 meV/atom')
ax2.set_xlabel('cutoff (eV)')
ax2.set_ylabel('|dE| (meV/atom)')
ax2.legend()
plt.tight_layout()
for c2, d in zip(cut[1:], de_meV):
    print(f"{c2:6.0f} eV: change {d:+8.3f} meV/atom")

# Provisional choice: first cutoff where |dE| stays below ~5 meV/atom thereafter.
# Update after inspecting the plot.
CUTOFF_PROVISIONAL = float(cut[np.where(np.abs(de_meV) < 5)[0][0] + 1]) if np.any(np.abs(de_meV) < 5) else float(cut[-1])
print(f"Suggested cutoff (auto): {CUTOFF_PROVISIONAL:.0f} eV — verify on the plot, then set CUTOFF below.")


**My chosen cutoff / 我选择的截断能:** ____ eV  

**Because / 理由:** successive |ΔE| below ____ meV/atom from this point onward.


In [ ]:
# >>> EDIT after 1.1 analysis <<<
CUTOFF = 500  # eV — replace with your converged value from the plot above
with open('settings.txt', 'w') as f:
    f.write(f"CUTOFF {CUTOFF}\n")
print(f"Recorded CUTOFF = {CUTOFF} eV in settings.txt")


### 1.2 K-point convergence / k 点收敛


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J pt_kpt
#!/usr/bin/python3
# Job 1.2: Pt bulk k-point sweep
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import bulk
from ase.calculators.castep import Castep

metal, adsorbate, a0 = open('system.txt').read().split()
a0 = float(a0)

settings = dict(line.split() for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
kmeshes = [4, 6, 8, 10, 12]

atoms = bulk(metal, 'fcc', a=a0, cubic=True)

with open('results_kpoints.txt', 'w') as f:
    for k in kmeshes:
        calc = Castep(directory=f'bulk_k_{k}', label=f'bulk_k_{k}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'SinglePoint'
        calc.cell.kpoint_mp_grid = f'{k} {k} {k}'
        atoms.calc = calc
        e = atoms.get_potential_energy()
        f.write(f"{k} {e/len(atoms)}\n")
        f.flush()
        print(f"mesh {k}x{k}x{k} -> {e/len(atoms):.6f} eV/atom", flush=True)
print("done")


In [ ]:
# Analyse k-point sweep
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('results_kpoints.txt')
k, e = data[:, 0], data[:, 1]
de_meV = np.diff(e) * 1000

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(k, e, 'o-')
ax1.set_xlabel('mesh k (k x k x k)')
ax1.set_ylabel('energy (eV/atom)')
ax2.semilogy(k[1:], np.abs(de_meV), 'o-')
ax2.axhline(5, ls='--', c='r', label='5 meV/atom')
ax2.set_xlabel('mesh k')
ax2.set_ylabel('|dE| (meV/atom)')
ax2.legend()
plt.tight_layout()
for k2, d in zip(k[1:], de_meV):
    print(f"{int(k2)}x{int(k2)}x{int(k2)}: change {d:+8.3f} meV/atom")


**My chosen bulk mesh / 体相网格:** ____ × ____ × ____  

**Linear-density note for the slab / 表面线密度：**  
bulk side $a_0 \approx$ ____ Å with mesh $n\times n\times n$.  
Slab is ~same in $x,y$ but ~30 Å in $z$ → use $n\times n\times 1$ (or denser in-plane if needed).


In [ ]:
# >>> EDIT after 1.2 analysis <<<
K_BULK = 8          # integer n for n x n x n
KMESH_BULK = f'{K_BULK} {K_BULK} {K_BULK}'
KMESH_SLAB = f'{K_BULK} {K_BULK} 1'   # same in-plane linear density; 1 along long z

with open('settings.txt', 'a') as f:
    f.write(f"KMESH_BULK {KMESH_BULK}\n")
    f.write(f"KMESH_SLAB {KMESH_SLAB}\n")
print(f"Bulk mesh {KMESH_BULK}; slab mesh {KMESH_SLAB}")


### 1.3 Lattice constant / 晶格常数


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J pt_latt
#!/usr/bin/python3
# Job 1.3: optimise Pt lattice constant
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import bulk
from ase.calculators.castep import Castep
from ase.io import read

metal, adsorbate, a0 = open('system.txt').read().split()
a0 = float(a0)
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_BULK'].strip()

atoms = bulk(metal, 'fcc', a=a0, cubic=True)
calc = Castep(directory='bulk_opt', label='bulk_opt',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.cell.kpoint_mp_grid = KMESH
calc.param.task = 'GeometryOptimization'
atoms.calc = calc
e = atoms.get_potential_energy()

final = read('bulk_opt/bulk_opt-out.cell')
a_opt = final.cell.lengths()[0]
with open('a0.txt', 'w') as f:
    f.write(f"{a_opt}\n")
# energy per atom for surface-energy formula later
with open('E_bulk_per_atom.txt', 'w') as f:
    f.write(f"{e/len(atoms)}\n")
print(f"optimised a0 = {a_opt:.4f} A (guess was {a0} A); E/atom = {e/len(atoms):.6f} eV")


**DFT $a_0$:** ____ Å  

**Experimental $a_0$ + source:** 3.924 Å (____ cite)  

**Comment:** PBE typically overestimates lattice constants by ~1–2% for late TMs — check your deviation.


## Part 2: Clean Pt(100) / 洁净表面

**ASE `fcc100` tags:** tag 1 = **top** (surface); largest tag = **bottom**. Freeze the bottom half.


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 8 -J pt_layers
#!/usr/bin/python3
# Job 2.1: layer convergence via surface energy
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import fcc100
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()

layer_list = [3, 4, 5, 6, 7]
VACUUM = 12  # ASE adds this above AND below

with open('results_layers.txt', 'w') as f:
    for nl in layer_list:
        slab = fcc100(metal, size=(1, 1, nl), a=a0, vacuum=VACUUM)
        frozen = [atom.index for atom in slab if atom.tag > nl / 2]
        slab.set_constraint(FixAtoms(indices=frozen))
        calc = Castep(directory=f'slab_{nl}L', label=f'slab_{nl}L',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'GeometryOptimization'
calc.param.fix_occupancy = False
# Robust SCF settings for metallic Pt slabs.
# Pt 金属 slab 的稳健电子收敛设置。
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
        calc.cell.kpoint_mp_grid = KMESH
        # Relax ions, but keep the simulation cell fixed.
        # 只优化原子；晶胞保持固定。
        calc.cell.fix_all_cell = True
        slab.calc = calc
        e = slab.get_potential_energy()
        area = slab.cell[0, 0] * slab.cell[1, 1]
        f.write(f"{nl} {e} {len(slab)} {area}\n")
        f.flush()
        print(f"{nl} layers: E = {e:.6f} eV ({len(frozen)} atoms frozen)", flush=True)
print("done")


In [ ]:
# Surface energy gamma
import numpy as np
import matplotlib.pyplot as plt

E_BULK_PER_ATOM = float(open('E_bulk_per_atom.txt').read())
data = np.loadtxt('results_layers.txt')
nl, e_slab, n_atoms, area = data.T
gamma_eV = (e_slab - n_atoms * E_BULK_PER_ATOM) / (2.0 * area)
gamma_J = gamma_eV * 16.0218

plt.plot(nl, gamma_J, 'o-')
plt.xlabel('layers')
plt.ylabel('surface energy (J/m$^2$)')
plt.title('Pt(100): gamma vs thickness')
plt.tight_layout()
for n, g in zip(nl, gamma_J):
    print(f"{int(n)} layers: gamma = {g:.3f} J/m^2")


In [ ]:
# >>> EDIT after gamma plot <<<
NL = 5
VACUUM = 12
with open('settings.txt', 'a') as f:
    f.write(f"NL {NL}\n")
    f.write(f"VACUUM {VACUUM}\n")
print(f"Using NL={NL}, VACUUM={VACUUM} (update VACUUM after 2.3 if needed)")


### 2.3 Vacuum convergence / 真空收敛


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 8 -J pt_vacuum
#!/usr/bin/python3
# Job 2.3: vacuum sweep at fixed thickness
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import fcc100
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()
NL = int(settings['NL'])

vacuum_list = [8, 10, 12, 14, 16]

with open('results_vacuum.txt', 'w') as f:
    for vac in vacuum_list:
        slab = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=vac)
        frozen = [atom.index for atom in slab if atom.tag > NL / 2]
        slab.set_constraint(FixAtoms(indices=frozen))
        calc = Castep(directory=f'slab_vac_{vac}', label=f'slab_vac_{vac}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'GeometryOptimization'
calc.param.fix_occupancy = False
# Robust SCF settings for metallic Pt slabs.
# Pt 金属 slab 的稳健电子收敛设置。
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
        calc.cell.kpoint_mp_grid = KMESH
        # Relax ions, but keep the simulation cell fixed.
        # 只优化原子；晶胞保持固定。
        calc.cell.fix_all_cell = True
        slab.calc = calc
        e = slab.get_potential_energy()
        f.write(f"{vac} {e}\n")
        f.flush()
        print(f"vacuum={vac}: E = {e:.6f} eV", flush=True)
print("done")


In [ ]:
# Plot vacuum convergence
import numpy as np
import matplotlib.pyplot as plt

vac, e = np.loadtxt('results_vacuum.txt').T
de = np.diff(e) * 1000  # meV
plt.plot(vac, e, 'o-')
plt.xlabel('ASE vacuum= (A)')
plt.ylabel('total energy (eV)')
plt.title('Vacuum convergence (fixed NL)')
plt.tight_layout()
for v, d in zip(vac[1:], de):
    print(f"vacuum {v}: dE = {d:+.2f} meV")


In [ ]:
# >>> EDIT after vacuum plot <<<
VACUUM = 12
# rewrite VACUUM line in settings (simple append-ok if you track latest manually)
with open('settings.txt', 'a') as f:
    f.write(f"VACUUM {VACUUM}\n")
print(f"Production VACUUM = {VACUUM}")


## Part 3: CO reference / 吸附物参考态

CO is a **closed-shell singlet**. Whole-molecule adsorption → $E_{\mathrm{ref}} = E(\mathrm{CO})$.
Also sweep CO cutoff and set production cutoff to $\max$(metal, CO).


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J co_ref
#!/usr/bin/python3
# Job 3.1: gas-phase CO in a large box
from environment_modules import module
module('load', 'compchem/castep')

from ase import Atoms
from ase.calculators.castep import Castep

settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])

# Isolated CO in a cubic box (Angstrom). Singlet: default spin for closed shell.
box = 12.0
co = Atoms('CO', positions=[[0, 0, 0], [0, 0, 1.14]], cell=[box]*3, pbc=True)
co.center()

calc = Castep(directory='co_ref', label='co_ref', castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.spin_polarized = False
calc.cell.kpoint_mp_grid = '1 1 1'
# Relax ions, but keep the simulation cell fixed.
# 只优化原子；晶胞保持固定。
calc.cell.fix_all_cell = True
co.calc = calc
e = co.get_potential_energy()
with open('results_reference.txt', 'w') as f:
    f.write(f"CO {e}\n")
print(f"E(CO) = {e:.6f} eV (singlet, box={box} A)")


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J co_cutoff
#!/usr/bin/python3
# Job 3.2: CO cutoff sweep (who is harder — Pt or C/O?)
from environment_modules import module
module('load', 'compchem/castep')

from ase import Atoms
from ase.calculators.castep import Castep

cutoffs = list(range(300, 701, 50))
box = 12.0

with open('results_ref_cutoff.txt', 'w') as f:
    for c in cutoffs:
        co = Atoms('CO', positions=[[0, 0, 0], [0, 0, 1.14]], cell=[box]*3, pbc=True)
        co.center()
        calc = Castep(directory=f'co_cut_{c}', label=f'co_cut_{c}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = c
        calc.param.task = 'SinglePoint'
        calc.param.spin_polarized = False
        calc.cell.kpoint_mp_grid = '1 1 1'
        co.calc = calc
        e = co.get_potential_energy()
        f.write(f"{c} {e}\n")
        f.flush()
        print(f"CO cutoff {c} -> {e:.6f} eV", flush=True)
print("done")


In [ ]:
# Decide production cutoff = max(metal, CO)
import numpy as np

metal_cut = float(dict(line.split(None, 1) for line in open('settings.txt'))['CUTOFF'])
data = np.loadtxt('results_ref_cutoff.txt')
cut, e = data[:, 0], data[:, 1]
de = np.abs(np.diff(e)) * 1000
for c2, d in zip(cut[1:], de):
    print(f"CO {c2:.0f} eV: |dE|={d:.3f} meV")

# Heuristic suggestion — verify on plots
co_sug = float(cut[np.where(de < 5)[0][0] + 1]) if np.any(de < 5) else float(cut[-1])
PROD = max(metal_cut, co_sug)
print(f"Metal cutoff {metal_cut:.0f}; CO suggested {co_sug:.0f}; PRODUCTION -> {PROD:.0f} eV")
print("If you change it, update settings.txt CUTOFF and re-run adsorbate-containing jobs.")


In [ ]:
# >>> EDIT: set production cutoff after comparing metal vs CO <<<
PRODUCTION_CUTOFF = 500  # eV
with open('settings.txt', 'a') as f:
    f.write(f"CUTOFF {PRODUCTION_CUTOFF}\n")  # latest CUTOFF wins when we read last
# Prefer a clean rewrite of key settings:
settings = {}
for line in open('settings.txt'):
    k, v = line.split(None, 1)
    settings[k] = v.strip()
settings['CUTOFF'] = str(PRODUCTION_CUTOFF)
with open('settings.txt', 'w') as f:
    for k, v in settings.items():
        f.write(f"{k} {v}\n")
print(open('settings.txt').read())


## Part 4: Adsorption of CO on Pt(100) — FULL REDO

**Strategy:** one structure per job, 16 cores, 4 h wall time. Fresh directories (`part4_*`) so old timed-out/`bak` files cannot interfere.

**Order (do not parallelise all four if the queue is busy):**
1. clean slab → `results_clean_slab.txt`
2. on-top → `result_ads_on-top.txt`
3. bridge → `result_ads_bridge.txt`
4. hollow → `result_ads_hollow.txt`
5. analysis cell → `results_Eads.txt` + fill geometry notes

**Production settings (must already be in `settings.txt` / `a0.txt`):**
PBE, 850 eV, a0 from Part 1.3, KMESH_SLAB 10 10 1, NL=5, VACUUM=12, E(CO) from Part 3.

**Optional cleanup on the cluster before starting** (in `~/submission/surface_study`):
```bash
# keep old junk aside; do not delete until you have new results
mkdir -p _old_part4_$(date +%Y%m%d)
mv slab_clean_final_850eV* ads_*_final_850eV* part4_* results_clean_slab.txt result_ads_*.txt results_ads_sites.txt results_Eads.txt _old_part4_$(date +%Y%m%d)/ 2>/dev/null || true
```


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 16 -J p4_clean
#!/usr/bin/python3
# Part 4.0 FROM SCRATCH: clean Pt(100) slab (production settings).
# Writes: results_clean_slab.txt
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import fcc100
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])

slab = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
# ASE fcc100: tag 1 = top surface; freeze bottom half (larger tags)
frozen = [atom.index for atom in slab if atom.tag > NL / 2]
slab.set_constraint(FixAtoms(indices=frozen))

calc = Castep(directory='part4_clean', label='part4_clean',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.spin_polarized = False
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
calc.cell.kpoint_mp_grid = KMESH
calc.cell.fix_all_cell = True
slab.calc = calc

e = slab.get_potential_energy()
open('results_clean_slab.txt', 'w').write(f'{e}\n')
print(f'E_slab = {e:.10f} eV')


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 16 -J p4_ot
#!/usr/bin/python3
# Part 4.1 FROM SCRATCH: CO on Pt(100) site = on-top
# Start: upright CO, C-down, height ~1.5 A, C-O = 1.14 A
# Writes: result_ads_on-top.txt and part4_ontop_final.xyz
from environment_modules import module
module('load', 'compchem/castep')

from ase import Atom
from ase.build import fcc100, add_adsorbate
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep
from ase.io import write

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])

slab = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
add_adsorbate(slab, 'C', height=1.5, position='ontop')
c_pos = slab.positions[-1].copy()
slab.append(Atom('O', c_pos + [0.0, 0.0, 1.14]))
frozen = [atom.index for atom in slab if atom.tag > NL / 2]
slab.set_constraint(FixAtoms(indices=frozen))

calc = Castep(directory='part4_ontop', label='part4_ontop',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.spin_polarized = False
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
calc.cell.kpoint_mp_grid = KMESH
calc.cell.fix_all_cell = True
slab.calc = calc

e = slab.get_potential_energy()
write('part4_ontop_final.xyz', slab)
open('result_ads_on-top.txt', 'w').write(f'{e}\n')
print(f'on-top: E = {e:.10f} eV')


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 16 -J p4_br
#!/usr/bin/python3
# Part 4.1 FROM SCRATCH: CO on Pt(100) site = bridge
# Start: upright CO, C-down, height ~1.5 A, C-O = 1.14 A
# Writes: result_ads_bridge.txt and part4_bridge_final.xyz
from environment_modules import module
module('load', 'compchem/castep')

from ase import Atom
from ase.build import fcc100, add_adsorbate
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep
from ase.io import write

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])

slab = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
add_adsorbate(slab, 'C', height=1.5, position='bridge')
c_pos = slab.positions[-1].copy()
slab.append(Atom('O', c_pos + [0.0, 0.0, 1.14]))
frozen = [atom.index for atom in slab if atom.tag > NL / 2]
slab.set_constraint(FixAtoms(indices=frozen))

calc = Castep(directory='part4_bridge', label='part4_bridge',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.spin_polarized = False
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
calc.cell.kpoint_mp_grid = KMESH
calc.cell.fix_all_cell = True
slab.calc = calc

e = slab.get_potential_energy()
write('part4_bridge_final.xyz', slab)
open('result_ads_bridge.txt', 'w').write(f'{e}\n')
print(f'bridge: E = {e:.10f} eV')


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 16 -J p4_ho
#!/usr/bin/python3
# Part 4.1 FROM SCRATCH: CO on Pt(100) site = hollow
# Start: upright CO, C-down, height ~1.5 A, C-O = 1.14 A
# Writes: result_ads_hollow.txt and part4_hollow_final.xyz
from environment_modules import module
module('load', 'compchem/castep')

from ase import Atom
from ase.build import fcc100, add_adsorbate
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep
from ase.io import write

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])

slab = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
add_adsorbate(slab, 'C', height=1.5, position='hollow')
c_pos = slab.positions[-1].copy()
slab.append(Atom('O', c_pos + [0.0, 0.0, 1.14]))
frozen = [atom.index for atom in slab if atom.tag > NL / 2]
slab.set_constraint(FixAtoms(indices=frozen))

calc = Castep(directory='part4_hollow', label='part4_hollow',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.spin_polarized = False
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
calc.cell.kpoint_mp_grid = KMESH
calc.cell.fix_all_cell = True
slab.calc = calc

e = slab.get_potential_energy()
write('part4_hollow_final.xyz', slab)
open('result_ads_hollow.txt', 'w').write(f'{e}\n')
print(f'hollow: E = {e:.10f} eV')


In [ ]:
# Part 4.2: adsorption energies (run after clean + three sites finish)
from pathlib import Path

E_slab = float(open('results_clean_slab.txt').read())
# results_reference.txt format from Part 3: usually "CO <energy>" or just energy
ref_toks = open('results_reference.txt').read().split()
E_ref = float(ref_toks[1]) if len(ref_toks) > 1 else float(ref_toks[0])

sites = [
    ('on-top', 'result_ads_on-top.txt'),
    ('bridge', 'result_ads_bridge.txt'),
    ('hollow', 'result_ads_hollow.txt'),
]

print(f'E_slab = {E_slab:.10f} eV')
print(f'E_ref  = {E_ref:.10f} eV  (E(CO))')
print(f'{"site":8s}  {"E_ads/slab (eV)":>16s}  {"E_ads (eV)":>12s}')

rows = []
best = None
missing = []
for site, path in sites:
    p = Path(path)
    if not p.is_file() or p.stat().st_size == 0:
        missing.append(path)
        continue
    e_as = float(p.read_text().split()[0])
    e_ads = e_as - E_slab - E_ref
    print(f'{site:8s}  {e_as:16.10f}  {e_ads:+12.6f}')
    rows.append((site, e_as, e_ads))
    if best is None or e_ads < best[1]:
        best = (site, e_ads)

if missing:
    raise SystemExit('Missing result files: ' + ', '.join(missing))

with open('results_ads_sites.txt', 'w') as f:
    for site, e_as, e_ads in rows:
        f.write(f'{site} {e_as}\n')
with open('results_Eads.txt', 'w') as f:
    for site, e_as, e_ads in rows:
        f.write(f'{site} {e_ads}\n')

print(f'Preferred site (most negative E_ads): {best[0]} ({best[1]:+.6f} eV)')


In [ ]:
# Part 4.3 helper: measure geometry from final XYZ (edit SITE if needed)
from ase.io import read
import numpy as np

SITE = 'ontop'  # 'ontop' | 'bridge' | 'hollow'
path = {
    'ontop': 'part4_ontop_final.xyz',
    'bridge': 'part4_bridge_final.xyz',
    'hollow': 'part4_hollow_final.xyz',
}[SITE]

atoms = read(path)
# CO: last two atoms are C then O in our builder
C = atoms[-2]
O = atoms[-1]
pt = [a for a in atoms if a.symbol == 'Pt']
# surface plane ~ mean z of top-layer Pt (smallest tag among Pt, tag==1)
tags = [a.tag for a in pt if a.tag > 0]
top_tag = min(tags) if tags else 1
top_pt = [a for a in pt if a.tag == top_tag]
z_surf = np.mean([a.position[2] for a in top_pt])
height_C = C.position[2] - z_surf
d_PtC = min(np.linalg.norm(C.position - a.position) for a in pt)
d_CO = np.linalg.norm(O.position - C.position)
v = O.position - C.position
tilt = np.degrees(np.arccos(np.clip(v[2] / np.linalg.norm(v), -1, 1)))

print(f'File: {path}')
print(f'Height of C above top Pt plane: {height_C:.3f} A')
print(f'Nearest Pt-C distance:          {d_PtC:.3f} A')
print(f'C-O bond length:                {d_CO:.3f} A')
print(f'Tilt from surface normal:       {tilt:.1f} deg (0 = upright)')


### 4.3 Geometry notes

**Preferred site:** ____

**Height of C above surface plane:** ____ Å

**Pt–C distance:** ____ Å

**C–O bond length:** ____ Å

**Orientation:** upright C-down / tilted? ____

**Did any site relax away from the start?** ____


## Part 5: Summary / 汇总

Fill after numbers exist. Discussion: pick any four questions from the class notebook end.


### Summary table / 汇总表

| Quantity | Value | Notes |
|---|---|---|
| Metal | Pt | FCC (100) |
| Adsorbate | CO | $E_{\mathrm{ref}}=E(\mathrm{CO})$, singlet |
| XC | PBE | |
| Production cutoff | ____ eV | set by ____ |
| Bulk k-mesh | ____ | |
| Slab k-mesh | ____ | linear density |
| Layers | ____ | from γ |
| Vacuum (ASE) | ____ Å | from E |
| $a_0$ (DFT) | ____ Å | exp 3.924 Å (source: ____) |
| $\gamma$(100) | ____ J/m² | |
| $E_{\mathrm{ads}}$ on-top | ____ eV | |
| $E_{\mathrm{ads}}$ bridge | ____ eV | |
| $E_{\mathrm{ads}}$ hollow | ____ eV | |
| Preferred site | ____ | |


### Discussion (4 answers) / 讨论四题

**Q1:** ____  
**A1:** ____  

**Q2:** ____  
**A2:** ____  

**Q3:** ____  
**A3:** ____  

**Q4:** ____  
**A4:** ____  


## Part 6: Extensions / 扩展（全部尝试；计分上限 +10）

Run order: **6.1 → 6.2 → 6.3 → 6.5 → 6.4**.


### 6.1 Diffusion barrier / 扩散势垒

Constrained path hollow → bridge → hollow for CO on Pt(100).


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 8 -J co_barrier
#!/usr/bin/python3
# Job 6.1: constrained lateral path hollow-bridge-hollow
from environment_modules import module
module('load', 'compchem/castep')

import numpy as np
from ase.build import fcc100, add_adsorbate
from ase.constraints import FixAtoms, FixInternals
from ase.calculators.castep import Castep
from ase import Atom

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])

# Sample fractional path coordinate s in [0, 1]: 0=hollow, 0.5=bridge, 1=next hollow
# For fcc100 (1x1), get Cartesian site positions from ASE helper slabs.
def site_xy(site):
    s = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
    add_adsorbate(s, 'C', height=1.5, position=site)
    return s.positions[-1][:2].copy()

xy_h1 = site_xy('hollow')
xy_br = site_xy('bridge')
# second hollow: translate by surface lattice vector a along x (approx for 1x1)
slab0 = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
a_vec = slab0.cell[0, :2]
xy_h2 = xy_h1 + a_vec

n_points = 7
ss = np.linspace(0.0, 1.0, n_points)

with open('results_diffusion.txt', 'w') as f:
    for i, s in enumerate(ss):
        if s <= 0.5:
            t = s / 0.5
            xy = (1 - t) * xy_h1 + t * xy_br
        else:
            t = (s - 0.5) / 0.5
            xy = (1 - t) * xy_br + t * xy_h2
        slab = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
        # place C at xy with starting height; O above
        # use hollow height reference
        add_adsorbate(slab, 'C', height=1.5, position='hollow')
        slab.positions[-1, 0] = xy[0]
        slab.positions[-1, 1] = xy[1]
        c_pos = slab.positions[-1].copy()
        slab.append(Atom('O', c_pos + [0, 0, 1.14]))
        # freeze bottom metal + fix C x,y (allow z and O to relax): use FixAtoms on C x,y via indices mask
        # Practical approach: freeze all metal bottom; constrain C lateral by FixAtoms on a dummy — 
        # ASE FixAtoms freezes all DOF. Instead freeze C completely for a rigid lateral scan of height-relaxed O only,
        # OR freeze metal bottom and C x,y by setting momenta/constraints carefully.
        # Here: freeze bottom metal layers AND freeze the C atom fully; relax O + top metal.
        # Better physics: use FixInternals / custom — keep it simple for a first barrier estimate:
        frozen = [atom.index for atom in slab if (atom.tag > NL / 2) or (atom.symbol == 'C')]
        slab.set_constraint(FixAtoms(indices=frozen))
        calc = Castep(directory=f'diff_{i:02d}', label=f'diff_{i:02d}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'GeometryOptimization'
calc.param.fix_occupancy = False
# Robust SCF settings for metallic Pt slabs.
# Pt 金属 slab 的稳健电子收敛设置。
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
        calc.cell.kpoint_mp_grid = KMESH
        # Relax ions, but keep the simulation cell fixed.
        # 只优化原子；晶胞保持固定。
        calc.cell.fix_all_cell = True
        slab.calc = calc
        e = slab.get_potential_energy()
        f.write(f"{s:.4f} {e:.8f}\n")
        f.flush()
        print(f"s={s:.3f}: E={e:.6f} eV", flush=True)
print("done")


In [ ]:
# Barrier from constrained path
import numpy as np
import matplotlib.pyplot as plt

s, e = np.loadtxt('results_diffusion.txt').T
e_rel = e - e.min()
plt.plot(s, e_rel, 'o-')
plt.xlabel('path coordinate s (0=hollow, 0.5=bridge, 1=hollow)')
plt.ylabel('E - E_min (eV)')
plt.title('CO diffusion path on Pt(100)')
plt.tight_layout()
Eb = e_rel.max()
print(f"Barrier estimate Eb ≈ {Eb:.3f} eV (max along path relative to minimum)")
print("Refine with denser points / better constraints if needed; discuss migration implications.")


### 6.2 ZPE / 零点能


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 8 -J co_zpe
#!/usr/bin/python3
# Job 6.2: finite-displacement vib for adsorbed CO (preferred site) + gas CO
# Uses ASE Vibrations. Prefer the site from results_Eads.txt (most negative).
from environment_modules import module
module('load', 'compchem/castep')

import numpy as np
from ase.build import fcc100, add_adsorbate
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep
from ase.vibrations import Vibrations
from ase import Atom, Atoms

settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_SLAB'].strip()
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])
metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())

# preferred site
rows = [line.split() for line in open('results_Eads.txt')]
pref = min(rows, key=lambda r: float(r[1]))[0]
ase_site = {'on-top': 'ontop', 'bridge': 'bridge', 'hollow': 'hollow'}[pref]

def make_calc(directory):
    calc = Castep(directory=directory, label=directory,
                  castep_command='srun castep.mpi')
    calc.param.xc_functional = 'PBE'
    calc.param.cut_off_energy = CUTOFF
    calc.param.task = 'SinglePoint'
    calc.cell.kpoint_mp_grid = KMESH
    # Relax ions, but keep the simulation cell fixed.
    # 只优化原子；晶胞保持固定。
    calc.cell.fix_all_cell = True
    return calc

# --- adsorbed CO ---
slab = fcc100(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
add_adsorbate(slab, 'C', height=1.5, position=ase_site)
c_pos = slab.positions[-1].copy()
slab.append(Atom('O', c_pos + [0, 0, 1.14]))
# freeze all metal; vibrate C and O only
frozen = [atom.index for atom in slab if atom.symbol == 'Pt']
slab.set_constraint(FixAtoms(indices=frozen))
slab.calc = make_calc('vib_ads')
# quick re-opt optional: skip if ads_* already optimised — for robustness do SinglePoint vib on current geometry
indices = [atom.index for atom in slab if atom.symbol in ('C', 'O')]
vib = Vibrations(slab, indices=indices, name='vib_ads_co')
vib.run()
vib.summary()
zpe_ads = vib.get_zero_point_energy()

# --- gas CO ---
box = 12.0
co = Atoms('CO', positions=[[0, 0, 0], [0, 0, 1.14]], cell=[box]*3, pbc=True)
co.center()
calc = Castep(directory='vib_co_gas', label='vib_co_gas', castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'SinglePoint'
calc.param.spin_polarized = False
calc.cell.kpoint_mp_grid = '1 1 1'
# Relax ions, but keep the simulation cell fixed.
# 只优化原子；晶胞保持固定。
calc.cell.fix_all_cell = True
co.calc = calc
vib2 = Vibrations(co, name='vib_gas_co')
vib2.run()
vib2.summary()
zpe_gas = vib2.get_zero_point_energy()

e_ads_el = float(min(rows, key=lambda r: float(r[1]))[1])
e_ads_zpe = e_ads_el + zpe_ads - zpe_gas
with open('results_zpe.txt', 'w') as f:
    f.write(f"site {pref}\n")
    f.write(f"ZPE_ads {zpe_ads}\n")
    f.write(f"ZPE_gas {zpe_gas}\n")
    f.write(f"Eads_el {e_ads_el}\n")
    f.write(f"Eads_zpe {e_ads_zpe}\n")
print(f"ZPE_ads={zpe_ads:.4f} eV; ZPE_gas={zpe_gas:.4f} eV")
print(f"E_ads electronic={e_ads_el:+.4f} eV; ZPE-corrected={e_ads_zpe:+.4f} eV")


### 6.3 Equation of state / 块体模量 $B_0$


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 4 -J pt_eos
#!/usr/bin/python3
# Job 6.3: E-V curve for FCC Pt
from environment_modules import module
module('load', 'compchem/castep')

import numpy as np
from ase.build import bulk
from ase.calculators.castep import Castep

metal, adsorbate, _ = open('system.txt').read().split()
a_ref = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
KMESH = settings['KMESH_BULK'].strip()

scales = np.linspace(0.94, 1.06, 7)

with open('results_eos.txt', 'w') as f:
    for s in scales:
        a = a_ref * s
        atoms = bulk(metal, 'fcc', a=a, cubic=True)
        calc = Castep(directory=f'eos_{s:.3f}', label=f'eos_{s:.3f}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'SinglePoint'
        calc.cell.kpoint_mp_grid = KMESH
        atoms.calc = calc
        e = atoms.get_potential_energy()
        V = atoms.get_volume()
        f.write(f"{a:.6f} {V:.6f} {e:.8f}\n")
        f.flush()
        print(f"a={a:.4f} V={V:.3f} E={e:.6f}", flush=True)
print("done")


In [ ]:
# Fit B0 (Birch-Murnaghan 3rd order, simple numpy fit)
import numpy as np
import matplotlib.pyplot as plt

a, V, E = np.loadtxt('results_eos.txt').T
# per atom
E = E / 4.0
V = V / 4.0

# polynomial fit E(V) around minimum as a quick B0 estimate:
# B0 = V * d2E/dV2 at minimum
coef = np.polyfit(V, E, 3)
# E = c3 V^3 + c2 V^2 + c1 V + c0
c3, c2, c1, c0 = coef
# find V0 near data min
V0 = V[np.argmin(E)]
d2 = 6 * c3 * V0 + 2 * c2
B0_eV_A3 = V0 * d2
B0_GPa = B0_eV_A3 * 160.21766208  # eV/A^3 -> GPa

plt.plot(V, E, 'o')
Vv = np.linspace(V.min(), V.max(), 100)
plt.plot(Vv, np.polyval(coef, Vv), '-')
plt.xlabel('volume / atom (A$^3$)')
plt.ylabel('energy / atom (eV)')
plt.title(f'Pt EOS — B0 ≈ {B0_GPa:.0f} GPa (polynomial estimate)')
plt.tight_layout()
print(f"B0 ≈ {B0_GPa:.1f} GPa (compare to experiment ~230 GPa for Pt; cite a source)")
print("Optional: replace with full Birch-Murnaghan fit from the class notebook method.")


### 6.5 Coverage / 低覆盖度（先于 6.4）


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 16 -J co_cover
#!/usr/bin/python3
# Job 6.5: CO on (2x2) Pt(100) at preferred site (lower coverage)
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import fcc100, add_adsorbate
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep
from ase import Atom

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])
# denser cell -> keep linear density: if 1x1 used n n 1, 2x2 uses (n/2) (n/2) 1
n = int(settings['KMESH_SLAB'].split()[0])
k2 = max(1, n // 2)
KMESH = f'{k2} {k2} 1'

rows = [line.split() for line in open('results_Eads.txt')]
pref = min(rows, key=lambda r: float(r[1]))[0]
ase_site = {'on-top': 'ontop', 'bridge': 'bridge', 'hollow': 'hollow'}[pref]

# clean 2x2
slab = fcc100(metal, size=(2, 2, NL), a=a0, vacuum=VACUUM)
frozen = [atom.index for atom in slab if atom.tag > NL / 2]
slab.set_constraint(FixAtoms(indices=frozen))
calc = Castep(directory='slab_clean_2x2', label='slab_clean_2x2',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.fix_occupancy = False
# Robust SCF settings for metallic Pt slabs.
# Pt 金属 slab 的稳健电子收敛设置。
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
calc.cell.kpoint_mp_grid = KMESH
# Relax ions, but keep the simulation cell fixed.
# 只优化原子；晶胞保持固定。
calc.cell.fix_all_cell = True
slab.calc = calc
e_slab = slab.get_potential_energy()

# ads 2x2
slab2 = fcc100(metal, size=(2, 2, NL), a=a0, vacuum=VACUUM)
add_adsorbate(slab2, 'C', height=1.5, position=ase_site)
c_pos = slab2.positions[-1].copy()
slab2.append(Atom('O', c_pos + [0, 0, 1.14]))
frozen = [atom.index for atom in slab2 if atom.tag > NL / 2]
slab2.set_constraint(FixAtoms(indices=frozen))
calc = Castep(directory='ads_2x2', label='ads_2x2', castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.fix_occupancy = False
# Robust SCF settings for metallic Pt slabs.
# Pt 金属 slab 的稳健电子收敛设置。
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
calc.cell.kpoint_mp_grid = KMESH
# Relax ions, but keep the simulation cell fixed.
# 只优化原子；晶胞保持固定。
calc.cell.fix_all_cell = True
slab2.calc = calc
e_as = slab2.get_potential_energy()

E_ref = float(open('results_reference.txt').read().split()[1])
e_ads = e_as - e_slab - E_ref
with open('results_coverage.txt', 'w') as f:
    f.write(f"site {pref}\n")
    f.write(f"kmesh {KMESH}\n")
    f.write(f"E_slab {e_slab}\n")
    f.write(f"E_ads_slab {e_as}\n")
    f.write(f"E_ads {e_ads}\n")
print(f"2x2 coverage E_ads ({pref}) = {e_ads:+.4f} eV with k={KMESH}")


### 6.4 Pt(111) comparison / (111) 对比（最后）


In [ ]:
%%sbatch --wait -p aqmtcm -A aqmtcm -t 240 -n 8 -J pt111_co
#!/usr/bin/python3
# Job 6.4: CO on Pt(111) — ontop / bridge / fcc hollow / hcp hollow
from environment_modules import module
module('load', 'compchem/castep')

from ase.build import fcc111, add_adsorbate
from ase.constraints import FixAtoms
from ase.calculators.castep import Castep
from ase import Atom

metal, adsorbate, _ = open('system.txt').read().split()
a0 = float(open('a0.txt').read())
settings = dict(line.split(None, 1) for line in open('settings.txt'))
CUTOFF = float(settings['CUTOFF'])
NL = int(settings['NL'])
VACUUM = float(settings['VACUUM'])
# hexagonal cell: use same in-plane density as 100 as a starting point
KMESH = settings['KMESH_SLAB'].strip()

sites = [('ontop', 'on-top'), ('bridge', 'bridge'), ('fcc', 'fcc'), ('hcp', 'hcp')]

# clean
slab = fcc111(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
frozen = [atom.index for atom in slab if atom.tag > NL / 2]
slab.set_constraint(FixAtoms(indices=frozen))
calc = Castep(directory='slab111_clean', label='slab111_clean',
              castep_command='srun castep.mpi')
calc.param.xc_functional = 'PBE'
calc.param.cut_off_energy = CUTOFF
calc.param.task = 'GeometryOptimization'
calc.param.fix_occupancy = False
# Robust SCF settings for metallic Pt slabs.
# Pt 金属 slab 的稳健电子收敛设置。
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
calc.cell.kpoint_mp_grid = KMESH
# Relax ions, but keep the simulation cell fixed.
# 只优化原子；晶胞保持固定。
calc.cell.fix_all_cell = True
slab.calc = calc
e_slab = slab.get_potential_energy()
E_ref = float(open('results_reference.txt').read().split()[1])

with open('results_ads_111.txt', 'w') as f:
    for ase_site, label in sites:
        s = fcc111(metal, size=(1, 1, NL), a=a0, vacuum=VACUUM)
        add_adsorbate(s, 'C', height=1.5, position=ase_site)
        c_pos = s.positions[-1].copy()
        s.append(Atom('O', c_pos + [0, 0, 1.14]))
        frozen = [atom.index for atom in s if atom.tag > NL / 2]
        s.set_constraint(FixAtoms(indices=frozen))
        calc = Castep(directory=f'ads111_{label}', label=f'ads111_{label}',
                      castep_command='srun castep.mpi')
        calc.param.xc_functional = 'PBE'
        calc.param.cut_off_energy = CUTOFF
        calc.param.task = 'GeometryOptimization'
calc.param.fix_occupancy = False
# Robust SCF settings for metallic Pt slabs.
# Pt 金属 slab 的稳健电子收敛设置。
calc.param.metals_method = 'EDFT'
calc.param.smearing_scheme = 'ColdSmearing'
calc.param.smearing_width = 0.1
calc.param.elec_energy_tol = 1e-6
calc.param.max_scf_cycles = 100
        calc.cell.kpoint_mp_grid = KMESH
        # Relax ions, but keep the simulation cell fixed.
        # 只优化原子；晶胞保持固定。
        calc.cell.fix_all_cell = True
        s.calc = calc
        e = s.get_potential_energy()
        e_ads = e - e_slab - E_ref
        f.write(f"{label} {e} {e_ads}\n")
        f.flush()
        print(f"111 {label}: E_ads = {e_ads:+.4f} eV", flush=True)
print("done — compare preferred site and E_ads to Pt(100)")


### Extension scoreboard / 扩展记分板

| Item | Max | Done | Notes |
|---|---:|:---:|---|
| 6.1 Barrier | 6 | [ ] | Eb = ____ eV |
| 6.2 ZPE | 5 | [ ] | E_ads(ZPE) = ____ eV |
| 6.3 B0 | 4 | [ ] | B0 = ____ GPa |
| 6.5 Coverage | 5 | [ ] | E_ads(2×2) = ____ eV |
| 6.4 Pt(111) | 8 | [ ] | pref = ____ |
| **Counted** | **≤10** |  | ____ |


## Submit / 提交

Copy to `~/submission/surface_study/` before the deadline:

- [ ] This notebook with saved outputs
- [ ] `system.txt`, `settings.txt`, `a0.txt`, `E_bulk_per_atom.txt`
- [ ] All `results_*.txt`
- [ ] Summary table + four discussion answers
- [ ] Cell 1: NAME, USERNAME, Pt, CO
